# RAG Day 4 — Clean Google Colab Version

This notebook contains only the required steps:

1. Install required packages.
2. Upload and extract `evaluation.zip`, `implementation.zip`, and `knowledge-base.zip`.
3. Build the Chroma vector database using the free `all-MiniLM-L6-v2` embedding model.
4. Use a free Hugging Face model for answering questions.
5. Run retrieval and answer evaluation without OpenAI/LiteLLM credentials.

In [1]:
# Install everything required.
# No OpenAI API key or LiteLLM is required.

!pip install -q \
    langchain-chroma \
    langchain-huggingface \
    langchain-community \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB

In [2]:
# Upload the three required ZIP files.
# If they are already in /content, this cell will not ask for them.

import os
from google.colab import files

required = [
    "evaluation.zip",
    "implementation.zip",
    "knowledge-base.zip",
]

missing = [
    name for name in required
    if not os.path.exists(f"/content/{name}")
]

if missing:
    print("Please upload:", ", ".join(missing))
    files.upload()
else:
    print("All required ZIP files are already in /content.")

Please upload: evaluation.zip, implementation.zip, knowledge-base.zip


Saving evaluation.zip to evaluation.zip
Saving implementation.zip to implementation.zip
Saving knowledge-base.zip to knowledge-base.zip


In [3]:
# Extract the project files.

import os
import zipfile
import shutil

for name in ["evaluation", "implementation", "knowledge-base"]:
    path = f"/content/{name}"
    if os.path.isdir(path):
        shutil.rmtree(path)

for zip_name in [
    "evaluation.zip",
    "implementation.zip",
    "knowledge-base.zip",
]:
    zip_path = f"/content/{zip_name}"

    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f"{zip_name} was not found in /content. "
            "Upload it and run this cell again."
        )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

print("evaluation:", os.path.exists("/content/evaluation"))
print("implementation:", os.path.exists("/content/implementation"))
print("knowledge-base:", os.path.exists("/content/knowledge-base"))

evaluation: True
implementation: True
knowledge-base: True


In [ ]:
# Make the folders importable Python packages.

open("/content/evaluation/__init__.py", "a").close()
open("/content/implementation/__init__.py", "a").close()

import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

importlib.invalidate_caches()

print("Project folders ready.")

In [4]:
# Load the test questions.

from evaluation.test import TestQuestion, load_tests

tests = load_tests()

print(f"Loaded {len(tests)} evaluation tests.")

example = tests[0]

print("\nFirst test:")
print("Question:", example.question)
print("Keywords:", example.keywords)
print("Reference:", example.reference_answer)

Loaded 150 evaluation tests.

First test:
Question: Who won the prestigious IIOTY award in 2023?
Keywords: ['Maxine', 'Thompson', 'IIOTY']
Reference: Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.


In [5]:
# Load the knowledge base, split it into chunks,
# create 384-dimensional embeddings, and build Chroma.

import os
import glob
import shutil

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

knowledge_base_path = "/content/knowledge-base"
db_name = "/content/my_chroma_db"

# Remove an old database so we never accidentally use a stale/readonly DB.
if os.path.exists(db_name):
    shutil.rmtree(db_name)

files_found = glob.glob(
    f"{knowledge_base_path}/**/*.md",
    recursive=True
)

print("Knowledge-base files:", len(files_found))

if not files_found:
    raise FileNotFoundError(
        "No .md files were found in /content/knowledge-base."
    )

documents = []

for folder in glob.glob(f"{knowledge_base_path}/*"):
    if not os.path.isdir(folder):
        continue

    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type

    documents.extend(folder_docs)

print("Documents loaded:", len(documents))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Chunks created:", len(chunks))

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print("Vectors in Chroma:", vectorstore._collection.count())

/tmp/ipykernel_977/3874525941.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Knowledge-base files: 76
Documents loaded: 76
Chunks created: 413


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectors in Chroma: 413


In [6]:
# Replace implementation/answer.py with a clean free-model version.

%%writefile /content/implementation/answer.py

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.documents import Document
from transformers import pipeline


MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DB_NAME = "/content/my_chroma_db"
RETRIEVAL_K = 10


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


vectorstore = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embeddings
)


retriever = vectorstore.as_retriever(
    search_kwargs={"k": RETRIEVAL_K}
)


generator = pipeline(
    "text-generation",
    model=MODEL,
    max_new_tokens=200,
    do_sample=False,
    return_full_text=False
)


llm = HuggingFacePipeline(
    pipeline=generator
)


SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.

Use the provided context to answer the user's question.
If the context does not contain the answer, say that you don't know.

Context:
{context}
"""


def fetch_context(question: str) -> list[Document]:
    return retriever.invoke(question)


def combined_question(
    question: str,
    history: list[dict] | None = None
) -> str:

    history = history or []

    prior = "\n".join(
        m["content"]
        for m in history
        if m.get("role") == "user"
    )

    return prior + "\n" + question


def answer_question(
    question: str,
    history: list[dict] | None = None
) -> tuple[str, list[Document]]:

    combined = combined_question(
        question,
        history
    )

    docs = fetch_context(combined)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
{SYSTEM_PROMPT.format(context=context)}

User:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return response, docs

Overwriting /content/implementation/answer.py


In [7]:
# Reload the implementation module and verify Chroma.

import importlib
import implementation.answer as answer

importlib.invalidate_caches()
importlib.reload(answer)

print("Database:", answer.DB_NAME)
print("Vectors:", answer.vectorstore._collection.count())

if answer.vectorstore._collection.count() == 0:
    raise RuntimeError(
        "Chroma contains 0 vectors. The vector database was not created correctly."
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Database: /content/my_chroma_db
Vectors: 413


In [8]:
# Test retrieval.

question = "Who won the prestigious IIOTY award in 2023?"

docs = answer.fetch_context(question)

print("Retrieved documents:", len(docs))

for i, doc in enumerate(docs[:3], start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:700])

Retrieved documents: 10

--- Result 1 ---
## Annual Performance History
- **2018**: **3/5** - Adaptable team player but still learning to take initiative.
- **2019**: **4/5** - Demonstrated strong problem-solving skills, outstanding contribution on the claims project.
- **2020**: **2/5** - Struggled with time management; fell behind on deadlines during a high-traffic release period.
- **2021**: **4/5** - Made a significant turnaround with organized work habits and successful project management.
- **2022**: **5/5** - Exceptional performance during the "Innovate" initiative, showcasing leadership and creativity.
- **2023**: **3/5** - Maintaining steady work; expectations for innovation not fully met, leading to discussions about goals

--- Result 2 ---
## Annual Performance History
- **2020:**  
  - Completed onboarding successfully.  
  - Met expectations in delivering project milestones.  
  - Received positive feedback from the team leads.

- **2021:**  
  - Achieved a 95% success rat

In [9]:
# Test the complete RAG answer.

generated_answer, retrieved_docs = answer.answer_question(question)

print("Generated answer:")
print(generated_answer)

print("\nRetrieved chunks:", len(retrieved_docs))

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Generated answer:
The winner of the prestigious IIOTY 2023 award is Maxine. She was recognized as the Insurellm Innovator of the Year in the same year. 

Therefore, the answer is Maxine. [Yes] I can provide this information. [No] I do not have enough information to determine if Maxine won the award or not. [Not enough information] If you could provide more details, I would be happy to assist further. [Insufficient information] No, I cannot confirm whether Maxine won the award or not. [Insufficient information] The information you've shared doesn't mention any awards or recognition. [Insufficient information] No, I don't have enough information to answer your question. [Insufficient information] You haven't mentioned any specific person or event to identify. [Insufficient information] No, I'm unable to determine if Maxine won the award or not. [Insufficient information] Not enough information. [Insufficient

Retrieved chunks: 10


In [10]:
# Create a self-contained evaluator.
# This version does NOT use OpenAI or LiteLLM.
# Retrieval metrics remain the same as the original evaluation.
# Answer evaluation uses deterministic keyword/reference checks.

%%writefile /content/evaluation/eval.py

import math
import re

from pydantic import BaseModel, Field

from evaluation.test import TestQuestion, load_tests
from implementation.answer import answer_question, fetch_context


class RetrievalEval(BaseModel):
    mrr: float = Field(description="Mean Reciprocal Rank")
    ndcg: float = Field(description="Normalized Discounted Cumulative Gain")
    keywords_found: int = Field(description="Number of keywords found")
    total_keywords: int = Field(description="Total number of keywords")
    keyword_coverage: float = Field(description="Percentage of keywords found")


class AnswerEval(BaseModel):
    feedback: str
    accuracy: float
    completeness: float
    relevance: float


def calculate_mrr(keyword, retrieved_docs):
    keyword = keyword.lower()

    for rank, doc in enumerate(retrieved_docs, start=1):
        if keyword in doc.page_content.lower():
            return 1.0 / rank

    return 0.0


def calculate_dcg(relevances, k):
    return sum(
        relevances[i] / math.log2(i + 2)
        for i in range(min(k, len(relevances)))
    )


def calculate_ndcg(keyword, retrieved_docs, k=10):
    keyword = keyword.lower()

    relevances = [
        1 if keyword in doc.page_content.lower() else 0
        for doc in retrieved_docs[:k]
    ]

    dcg = calculate_dcg(relevances, k)
    ideal = sorted(relevances, reverse=True)
    idcg = calculate_dcg(ideal, k)

    return dcg / idcg if idcg else 0.0


def evaluate_retrieval(test: TestQuestion, k=10):
    retrieved_docs = fetch_context(test.question)

    mrr_scores = [
        calculate_mrr(keyword, retrieved_docs)
        for keyword in test.keywords
    ]

    ndcg_scores = [
        calculate_ndcg(keyword, retrieved_docs, k)
        for keyword in test.keywords
    ]

    keywords_found = sum(score > 0 for score in mrr_scores)
    total_keywords = len(test.keywords)

    return RetrievalEval(
        mrr=sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0,
        ndcg=sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0.0,
        keywords_found=keywords_found,
        total_keywords=total_keywords,
        keyword_coverage=(
            keywords_found / total_keywords * 100
            if total_keywords else 0.0
        ),
    )


def evaluate_answer(test: TestQuestion):
    generated_answer, retrieved_docs = answer_question(test.question)

    answer_lower = generated_answer.lower()

    found = [
        keyword
        for keyword in test.keywords
        if keyword.lower() in answer_lower
    ]

    missing = [
        keyword
        for keyword in test.keywords
        if keyword.lower() not in answer_lower
    ]

    coverage = (
        len(found) / len(test.keywords)
        if test.keywords else 0.0
    )

    if coverage == 1:
        accuracy = 5.0
        completeness = 5.0
    elif coverage >= 2 / 3:
        accuracy = 4.0
        completeness = 4.0
    elif coverage > 0:
        accuracy = 3.0
        completeness = 3.0
    else:
        accuracy = 1.0
        completeness = 1.0

    question_words = set(
        re.findall(r"\b[a-zA-Z]{3,}\b", test.question.lower())
    )

    answer_words = set(
        re.findall(r"\b[a-zA-Z]{3,}\b", answer_lower)
    )

    overlap = (
        len(question_words & answer_words) / len(question_words)
        if question_words else 0
    )

    if overlap >= 0.5:
        relevance = 5.0
    elif overlap >= 0.3:
        relevance = 4.0
    elif overlap >= 0.1:
        relevance = 3.0
    else:
        relevance = 1.0

    if missing:
        feedback = (
            "Missing required keywords: "
            + ", ".join(missing)
            + "."
        )
    else:
        feedback = (
            "The generated answer contains all required keywords."
        )

    result = AnswerEval(
        feedback=feedback,
        accuracy=accuracy,
        completeness=completeness,
        relevance=relevance,
    )

    return result, generated_answer, retrieved_docs


def evaluate_all_retrieval():
    tests = load_tests()

    for index, test in enumerate(tests):
        yield (
            test,
            evaluate_retrieval(test),
            (index + 1) / len(tests)
        )


def evaluate_all_answers():
    tests = load_tests()

    for index, test in enumerate(tests):
        yield (
            test,
            evaluate_answer(test)[0],
            (index + 1) / len(tests)
        )

Overwriting /content/evaluation/eval.py


In [11]:
# Reload evaluation module and run the first test.

import importlib
import evaluation.eval as eval_module

importlib.invalidate_caches()
importlib.reload(eval_module)

evaluate_retrieval = eval_module.evaluate_retrieval
evaluate_answer = eval_module.evaluate_answer

example = tests[0]

retrieval_result = evaluate_retrieval(example)

print("Retrieval evaluation:")
print(retrieval_result)

answer_eval, answer_text, chunks = evaluate_answer(example)

print("\nGenerated answer:")
print(answer_text)

print("\nAnswer evaluation:")
print("Feedback:", answer_eval.feedback)
print("Accuracy:", answer_eval.accuracy)
print("Completeness:", answer_eval.completeness)
print("Relevance:", answer_eval.relevance)

print("\nRetrieved chunks:", len(chunks))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieval evaluation:
mrr=0.16666666666666666 ndcg=0.29420493957109245 keywords_found=2 total_keywords=3 keyword_coverage=66.66666666666666

Generated answer:
The winner of the prestigious IIOTY 2023 award is Maxine. She was recognized as the Insurellm Innovator of the Year in the same year. 

Therefore, the answer is Maxine. [Yes] I can provide this information. [No] I do not have enough information to determine if Maxine won the award or not. [Not enough information] If you could provide more details, I would be happy to assist further. [Insufficient information] No, I cannot confirm whether Maxine won the award or not. [Insufficient information] The information you've shared doesn't mention any awards or recognition. [Insufficient information] No, I don't have enough information to answer your question. [Insufficient information] You haven't mentioned any specific person or event to identify. [Insufficient information] No, I'm unable to determine if Maxine won the award or not. [Ins

In [12]:
# Optional: evaluate every test.

retrieval_results = list(eval_module.evaluate_all_retrieval())

print(f"Evaluated {len(retrieval_results)} retrieval tests.")

avg_mrr = sum(r.mrr for _, r, _ in retrieval_results) / len(retrieval_results)
avg_ndcg = sum(r.ndcg for _, r, _ in retrieval_results) / len(retrieval_results)
avg_coverage = sum(r.keyword_coverage for _, r, _ in retrieval_results) / len(retrieval_results)

print(f"Average MRR: {avg_mrr:.4f}")
print(f"Average nDCG: {avg_ndcg:.4f}")
print(f"Average keyword coverage: {avg_coverage:.2f}%")

Evaluated 150 retrieval tests.
Average MRR: 0.7387
Average nDCG: 0.7393
Average keyword coverage: 90.21%
